In [1]:
!pip install torch joblib matplotlib seaborn tqdm pgmpy fairlearn optuna

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 31.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.8/163.8 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 27.0 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: scipy
    Found existing installation: scipy 1.16.3
    Uninstalling scipy-1.16.3:
      Successfully uninstalled scipy-1.16.3


In [2]:
import os, gc, copy, math, warnings, logging
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

from sklearn.metrics import accuracy_score
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.isotonic import IsotonicRegression

import joblib
from tqdm import tqdm

from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.estimators import BayesianEstimator
from pgmpy.inference import VariableElimination

from fairlearn.metrics import demographic_parity_difference, equalized_odds_difference

warnings.filterwarnings('ignore')
logging.getLogger("pgmpy").setLevel(logging.ERROR)

SEED        = 42
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CACHE_DIR   = './cache'
RESULTS_DIR = '/kaggle/working'
BBN_MAX_ROWS    = 2_000
MAX_SWEEP_PAIRS = 200_000

# ─────────────────────────────────────────────────────────────
# PIPELINE MODE: set to 'eo_only' or 'dp_only' or 'both'
# For research comparison, run this script twice with different modes.
# ─────────────────────────────────────────────────────────────
PIPELINE_MODE = 'eo_only'   # <── change to 'dp_only' for the second run

for _d in [CACHE_DIR, RESULTS_DIR]:
    os.makedirs(_d, exist_ok=True)

def floor2(x: float) -> float:
    return math.floor(abs(float(x)) * 100) / 100

def set_seed(s: int = SEED):
    torch.manual_seed(s)
    np.random.seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)

set_seed()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

print(f"Device: {DEVICE}  |  CUDA: {torch.cuda.is_available()}")
print(f"Pipeline mode: {PIPELINE_MODE}")


# ─────────────────────────────────────────────────────────────
# KEY CHANGES vs v2/v3:
#
# PROBLEM IDENTIFIED in v2 logs:
#   Adult:    pre-sweep EO=0.1042 → needs tau=0.958 to get EO_fl=0.00
#             This is the root cause of the -6.3% accuracy drop.
#   COMPAS:   same pattern — tau=0.706 is aggressive
#   Hospital: tau=0.790 → -6.5% drop
#
#   The sweep is compensating for a model that still has significant
#   EO at the default threshold. FIX: make the model itself closer
#   to EO=0.00 pre-sweep so the sweep only needs fine-tuning.
#
# CHANGES:
#   1. EO_ONLY_MODE: when active, removes DP-adversary loss entirely
#      and doubles eo_loss_weight. The adversary only fights EO now.
#      In dp_only mode, the reverse applies.
#
#   2. Per-group probability calibration (GroupCalibrator):
#      After isotonic calibration, apply a second pass that nudges
#      the group-conditional score distributions toward the same
#      mean positive rate *within each label* (i.e., equalises TPR
#      and FPR directly in probability space before the sweep).
#      This is the main new mechanism for reducing sweep dependence.
#
#   3. Sweep budget reallocation: the per-group sweep now limits
#      tau range to [0.2, 0.8] by default (configurable per dataset)
#      to prevent extreme thresholds that destroy accuracy.
#      A fallback to the full range only happens if the constrained
#      sweep fails to achieve EO_fl=0.00.
#
#   4. Diagnostic script (run_diagnostics): measures and prints
#      the pre-sweep, post-calibration, and post-sweep EO/DP/acc
#      for every checkpoint, identifying where accuracy is lost.
#
#   5. PIPELINE_MODE flag: 'eo_only' / 'dp_only' / 'both'
#      Controls which fairness losses are active and which metric
#      is used as the primary sweep objective.
# ─────────────────────────────────────────────────────────────


@dataclass
class DatasetHParams:
    hidden_dim:           int   = 256
    fairness_dim:         int   = 128
    dropout:              float = 0.15
    lr:                   float = 1e-3
    batch_size:           int   = 64
    encoder_epochs:       int   = 200
    early_patience:       int   = 20
    bbn_prior_weight:     float = 0.20
    fairness_epochs:      int   = 300
    encoder_lr_factor:    float = 0.50
    adversary_lr_factor:  float = 1.00
    lambda_adv_start:     float = 0.30
    lambda_adv_max:       float = 3.00
    lambda_warmup_epochs: int   = 20
    cls_loss_weight_s2:   float = 0.50
    max_acc_drop:         float = 0.050
    stage2_max_acc_drop:  float = 0.055
    acc_drop_fallback:    float = 0.070
    use_bbn:              bool  = True
    bbn_weight:           float = 0.30
    bbn_threshold:        float = 0.28
    bbn_fairness_dir:     bool  = True
    use_isotonic:         bool  = True
    use_group_threshold:  bool  = True
    group_thresh_eps:     float = 0.75
    fine_w:               float = 0.35
    tau:                  float = 0.55
    bbn_adv_weight:       float = 0.20
    eo_loss_weight:       float = 1.50
    eo_smooth_temp:       float = 10.0
    adaptive_lambda:      bool  = True
    adaptive_lambda_clip: float = 4.0
    # NEW: constrain sweep tau range to avoid extreme thresholds
    sweep_tau_min:        float = 0.10   # don't go below this in primary sweep
    sweep_tau_max:        float = 0.90   # don't go above this in primary sweep
    # NEW: group calibration strength (0 = off, 1 = full correction)
    group_calib_alpha:    float = 0.40


DATASET_HPARAMS: Dict[str, DatasetHParams] = {

    "adult": DatasetHParams(
        hidden_dim=128, fairness_dim=64, dropout=0.25,
        lr=2.132e-4, batch_size=512,
        encoder_epochs=250, early_patience=25, bbn_prior_weight=0.20,
        fairness_epochs=500, encoder_lr_factor=0.70, adversary_lr_factor=1.5,
        lambda_adv_start=0.3, lambda_adv_max=1.5, lambda_warmup_epochs=20,
        cls_loss_weight_s2=0.7,
        max_acc_drop=0.07, stage2_max_acc_drop=0.08, acc_drop_fallback=0.10,
        use_bbn=True, bbn_weight=0.30, bbn_threshold=0.28, bbn_fairness_dir=True,
        use_isotonic=True, use_group_threshold=True,
        group_thresh_eps=0.75, fine_w=0.20, tau=0.45,
        bbn_adv_weight=0.20,
        eo_loss_weight=2.5,   # boosted from 2.0 — reduce sweep dependence
        eo_smooth_temp=10.0,
        adaptive_lambda=True, adaptive_lambda_clip=3.0,
        sweep_tau_min=0.30, sweep_tau_max=0.75,  # constrained — key change for Adult
        group_calib_alpha=0.50,
    ),

    "compas": DatasetHParams(
        hidden_dim=256, fairness_dim=128, dropout=0.15,
        lr=7.404e-4, batch_size=256,
        encoder_epochs=350, early_patience=25, bbn_prior_weight=0.20,
        fairness_epochs=800, encoder_lr_factor=0.0, adversary_lr_factor=1.2,
        lambda_adv_start=0.5, lambda_adv_max=6.0, lambda_warmup_epochs=5,
        cls_loss_weight_s2=0.5,
        max_acc_drop=0.10, stage2_max_acc_drop=0.11, acc_drop_fallback=0.14,
        use_bbn=True, bbn_weight=0.40, bbn_threshold=0.35, bbn_fairness_dir=True,
        use_isotonic=True, use_group_threshold=True,
        group_thresh_eps=0.65, fine_w=0.40, tau=0.45,
        bbn_adv_weight=0.20,
        eo_loss_weight=3.0,   # boosted from 2.5
        eo_smooth_temp=8.0,
        adaptive_lambda=True, adaptive_lambda_clip=4.0,
        sweep_tau_min=0.35, sweep_tau_max=0.80,  # constrained
        group_calib_alpha=0.55,
    ),

    "german": DatasetHParams(
        hidden_dim=192, fairness_dim=96, dropout=0.15,
        lr=2.864e-4, batch_size=512,
        encoder_epochs=400, early_patience=25, bbn_prior_weight=0.20,
        fairness_epochs=600, encoder_lr_factor=0.60, adversary_lr_factor=1.0,
        lambda_adv_start=0.4, lambda_adv_max=4.0, lambda_warmup_epochs=10,
        cls_loss_weight_s2=0.40,
        max_acc_drop=0.06, stage2_max_acc_drop=0.07, acc_drop_fallback=0.10,
        use_bbn=True, bbn_weight=0.35, bbn_threshold=0.45, bbn_fairness_dir=True,
        use_isotonic=False, use_group_threshold=True,
        group_thresh_eps=0.55, fine_w=0.20, tau=0.50,
        bbn_adv_weight=0.20,
        eo_loss_weight=1.5, eo_smooth_temp=6.0,
        adaptive_lambda=True, adaptive_lambda_clip=3.0,
        sweep_tau_min=0.15, sweep_tau_max=0.85,
        group_calib_alpha=0.35,
    ),

    "bank": DatasetHParams(
        hidden_dim=192, fairness_dim=96, dropout=0.15,
        lr=3.877e-4, batch_size=512,
        encoder_epochs=100, early_patience=25, bbn_prior_weight=0.20,
        fairness_epochs=150, encoder_lr_factor=0.70, adversary_lr_factor=1.25,
        lambda_adv_start=0.6, lambda_adv_max=7.5, lambda_warmup_epochs=10,
        cls_loss_weight_s2=0.4,
        max_acc_drop=0.08, stage2_max_acc_drop=0.09, acc_drop_fallback=0.10,
        use_bbn=True, bbn_weight=0.20, bbn_threshold=0.18, bbn_fairness_dir=False,
        use_isotonic=False, use_group_threshold=True,
        group_thresh_eps=0.40, fine_w=0.30, tau=0.55,
        bbn_adv_weight=0.20,
        eo_loss_weight=0.8, eo_smooth_temp=10.0,
        adaptive_lambda=False, adaptive_lambda_clip=2.0,
        sweep_tau_min=0.30, sweep_tau_max=0.75,
        group_calib_alpha=0.30,
    ),

    "lawschool": DatasetHParams(
        hidden_dim=192, fairness_dim=96, dropout=0.15,
        lr=2.864e-4, batch_size=512,
        encoder_epochs=150, early_patience=25, bbn_prior_weight=0.20,
        fairness_epochs=300, encoder_lr_factor=0.60, adversary_lr_factor=1.2,
        lambda_adv_start=0.4, lambda_adv_max=4.0, lambda_warmup_epochs=5,
        cls_loss_weight_s2=0.35,
        max_acc_drop=0.06, stage2_max_acc_drop=0.07, acc_drop_fallback=0.10,
        use_bbn=True, bbn_weight=0.25, bbn_threshold=0.18, bbn_fairness_dir=True,
        use_isotonic=False, use_group_threshold=True,
        group_thresh_eps=0.60, fine_w=0.20, tau=0.50,
        bbn_adv_weight=0.15,
        eo_loss_weight=1.8, eo_smooth_temp=8.0,
        adaptive_lambda=True, adaptive_lambda_clip=3.0,
        sweep_tau_min=0.20, sweep_tau_max=0.80,
        group_calib_alpha=0.30,
    ),

    "hospital": DatasetHParams(
        hidden_dim=128, fairness_dim=64, dropout=0.25,
        lr=1.134e-3, batch_size=128,
        encoder_epochs=200, early_patience=25, bbn_prior_weight=0.22,
        fairness_epochs=300, encoder_lr_factor=0.60, adversary_lr_factor=1.0,
        lambda_adv_start=0.30, lambda_adv_max=5.0, lambda_warmup_epochs=10,
        cls_loss_weight_s2=0.30,
        max_acc_drop=0.08, stage2_max_acc_drop=0.09, acc_drop_fallback=0.10,
        use_bbn=True, bbn_weight=0.22, bbn_threshold=0.18, bbn_fairness_dir=False,
        use_isotonic=True, use_group_threshold=True,
        group_thresh_eps=0.20, fine_w=0.50, tau=0.55,
        bbn_adv_weight=0.22,
        eo_loss_weight=1.8,   # boosted from 1.2 — reduce sweep dependence
        eo_smooth_temp=10.0,
        adaptive_lambda=True, adaptive_lambda_clip=3.0,
        sweep_tau_min=0.35, sweep_tau_max=0.75,  # constrained — key change for Hospital
        group_calib_alpha=0.45,
    ),
}

DATASET_CONFIG: Dict[str, Dict] = {
    "adult":     {"sens_attrs": [("sens_sex_train",     "sens_sex_test"),
                                 ("sens_race_train",    "sens_race_test")]},
    "compas":    {"sens_attrs": [("sens_race_train",    "sens_race_test"),
                                 ("sens_sex_train",     "sens_sex_test")]},
    "german":    {"sens_attrs": [("sens_age_train",     "sens_age_test"),
                                 ("sens_sex_train",     "sens_sex_test")]},
    "bank":      {"sens_attrs": [("sens_marital_train", "sens_marital_test"),
                                 ("sens_job_train",     "sens_job_test")]},
    "lawschool": {"sens_attrs": [("sens_race_train",    "sens_race_test"),
                                 ("sens_gender_train",  "sens_gender_test")]},
    "hospital":  {"sens_attrs": [("sens_race_train",    "sens_race_test"),
                                 ("sens_sex_train",     "sens_sex_test")]},
}

print(f"Hyperparams defined. Mode: {PIPELINE_MODE}")


# ─────────────────────────────────────────────────────────────
# NOVEL COMPONENT 1: Differentiable EO Surrogate Loss (unchanged)
# ─────────────────────────────────────────────────────────────

def differentiable_eo_loss(
    probs: torch.Tensor,
    y:     torch.Tensor,
    s:     torch.Tensor,
    temp:  float = 10.0,
    eps:   float = 1e-3,
) -> torch.Tensor:
    s_soft = torch.sigmoid(temp * (s - 0.5))
    s_neg  = 1.0 - s_soft
    y_pos  = y.float()
    y_neg  = 1.0 - y_pos

    def _smooth_rate(group_mask, label_mask):
        denom = (group_mask * label_mask).sum() + eps
        numer = (group_mask * label_mask * probs).sum()
        return numer / denom

    tpr0 = _smooth_rate(s_neg,  y_pos)
    tpr1 = _smooth_rate(s_soft, y_pos)
    fpr0 = _smooth_rate(s_neg,  y_neg)
    fpr1 = _smooth_rate(s_soft, y_neg)

    def _smooth_abs(x):
        return torch.sqrt(x**2 + eps)

    eo = _smooth_abs(tpr0 - tpr1) + _smooth_abs(fpr0 - fpr1)
    return eo


# ─────────────────────────────────────────────────────────────
# NEW: Differentiable DP Surrogate Loss
# Symmetric to the EO loss but only uses overall positive rates.
# Used when PIPELINE_MODE == 'dp_only'.
# ─────────────────────────────────────────────────────────────

def differentiable_dp_loss(
    probs: torch.Tensor,
    s:     torch.Tensor,
    temp:  float = 10.0,
    eps:   float = 1e-3,
) -> torch.Tensor:
    s_soft = torch.sigmoid(temp * (s - 0.5))
    s_neg  = 1.0 - s_soft

    def _smooth_rate(group_mask):
        denom = group_mask.sum() + eps
        numer = (group_mask * probs).sum()
        return numer / denom

    pr0 = _smooth_rate(s_neg)
    pr1 = _smooth_rate(s_soft)

    def _smooth_abs(x):
        return torch.sqrt(x**2 + eps)

    return _smooth_abs(pr0 - pr1)


# ─────────────────────────────────────────────────────────────
# NOVEL COMPONENT 2: Adaptive Lambda Scheduler (unchanged)
# ─────────────────────────────────────────────────────────────

class AdaptiveLambdaScheduler:
    def __init__(self, start: float, max_val: float, warmup: int,
                 adaptive: bool = True, clip: float = 4.0):
        self.start    = start
        self.max_val  = max_val
        self.warmup   = warmup
        self.adaptive = adaptive
        self.clip     = clip
        self._current_metric = 1.0

    def update_metric(self, val: float):
        self._current_metric = float(val)

    # kept for backward compatibility
    def update_eo(self, eo: float):
        self.update_metric(eo)

    def get(self, epoch: int) -> float:
        base = self.start + (self.max_val - self.start) * min(1.0, epoch / max(1, self.warmup))
        if not self.adaptive:
            return base
        scale = min(self.clip, max(1.0, self._current_metric / 0.005))
        if self._current_metric < 0.02:
            scale = min(scale, 1.5)
        return base * scale


# ─────────────────────────────────────────────────────────────
# NOVEL COMPONENT 3: EO-Guided BBN Soft Targets (unchanged)
# ─────────────────────────────────────────────────────────────

def bbn_eo_guided_targets(
    y_train:      np.ndarray,
    s_train_list: List[np.ndarray],
    bbn_probs:    np.ndarray,
    alpha:        float = 0.25,
) -> np.ndarray:
    soft = y_train.astype(float).copy()
    if not s_train_list:
        return soft
    s = s_train_list[0].astype(int)
    groups = np.unique(s)
    if len(groups) != 2:
        return soft
    g0, g1 = groups[0], groups[1]

    def _rate(mask, label):
        sub = bbn_probs[(s == mask) & (y_train == label)]
        return sub.mean() if len(sub) > 0 else 0.5

    tpr0 = _rate(g0, 1); tpr1 = _rate(g1, 1)
    fpr0 = _rate(g0, 0); fpr1 = _rate(g1, 0)

    for i in range(len(y_train)):
        gi = s[i]
        if y_train[i] == 1:
            tpr_excess = (tpr0 - tpr1) if gi == g0 else (tpr1 - tpr0)
            correction = alpha * max(0.0, tpr_excess)
            soft[i] = max(0.05, soft[i] - correction)
        else:
            fpr_excess = (fpr0 - fpr1) if gi == g0 else (fpr1 - fpr0)
            correction = alpha * max(0.0, fpr_excess)
            soft[i] = min(0.95, soft[i] + correction)
    return soft


# ─────────────────────────────────────────────────────────────
# NEW COMPONENT: Group Probability Calibrator
# ─────────────────────────────────────────────────────────────
# After isotonic calibration, the raw probability scores still
# have group-conditional TPR/FPR gaps. This component applies
# a second-pass linear correction that shifts the score
# distribution for each group toward equal TPR and FPR.
#
# Method: for each group g and label class c, compute the
# mean score delta needed to match the target rate, then
# apply alpha * delta as a soft push.
#
# This is the key mechanism that reduces sweep dependence —
# if the group calibrator does its job, the EO gap at tau=0.5
# should be near zero, and the sweep only needs fine-tuning.
# ─────────────────────────────────────────────────────────────

class GroupCalibrator:
    """
    Learns a per-group, per-label shift to equalise TPR and FPR
    in probability space. Applied AFTER isotonic calibration.
    Fitted on train probabilities, applied to test.
    """
    def __init__(self, alpha: float = 0.40):
        self.alpha = alpha
        self.shifts: Dict = {}  # (group, label) -> shift
        self.fitted = False

    def fit(self, proba_train: np.ndarray, y_train: np.ndarray,
            s_train: np.ndarray) -> 'GroupCalibrator':
        groups = np.unique(s_train)
        if len(groups) != 2:
            return self

        # Compute per-group, per-label mean scores
        means = {}
        for g in groups:
            for lbl in [0, 1]:
                mask = (s_train == g) & (y_train == lbl)
                means[(g, lbl)] = proba_train[mask].mean() if mask.sum() > 0 else 0.5

        # Target = average across groups (ensures fairness, not a hard 0/1)
        target_pos = np.mean([means[(g, 1)] for g in groups])
        target_neg = np.mean([means[(g, 0)] for g in groups])

        for g in groups:
            self.shifts[(g, 1)] = self.alpha * (target_pos - means[(g, 1)])
            self.shifts[(g, 0)] = self.alpha * (target_neg - means[(g, 0)])

        self.groups = groups
        self.fitted = True
        return self

    def transform(self, proba: np.ndarray, y_ref: np.ndarray,
                  s: np.ndarray) -> np.ndarray:
        """y_ref: used to select which shift to apply (0 or 1).
        At test time, use predicted labels as y_ref."""
        if not self.fitted:
            return proba
        out = proba.copy()
        for g in self.groups:
            for lbl in [0, 1]:
                mask = (s == g) & (y_ref == lbl)
                if mask.sum() == 0:
                    continue
                shift = self.shifts.get((g, lbl), 0.0)
                out[mask] = np.clip(out[mask] + shift, 0.01, 0.99)
        return out

    def transform_test(self, proba: np.ndarray, s: np.ndarray) -> np.ndarray:
        """Test-time: use predicted label (thresholded at 0.5) as y_ref."""
        y_pred = (proba >= 0.5).astype(int)
        return self.transform(proba, y_pred, s)


# ─────────────────────────────────────────────────────────────
# Utility helpers (unchanged)
# ─────────────────────────────────────────────────────────────

def to_dense(X) -> np.ndarray:
    arr = X.toarray() if hasattr(X, "toarray") else np.asarray(X)
    return np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)

def clean_numeric_column(s: pd.Series) -> pd.Series:
    s = pd.to_numeric(s, errors='coerce').replace([np.inf, -np.inf], np.nan)
    med = s.median()
    return s.fillna(0.0 if np.isnan(med) else med)

def safe_qcut(s: pd.Series, q: int = 5) -> pd.Series:
    s = clean_numeric_column(s)
    if s.nunique() <= 1:
        return pd.Series(0, index=s.index, dtype=int)
    try:
        return pd.qcut(s, q, labels=False, duplicates='drop').fillna(0).astype(int)
    except Exception:
        try:
            return pd.cut(s, q, labels=False, duplicates='drop').fillna(0).astype(int)
        except Exception:
            return pd.Series(0, index=s.index, dtype=int)

def make_num_pipeline():
    return Pipeline([('imp', SimpleImputer(strategy='median')),
                     ('sc',  StandardScaler())])

def make_cat_pipeline():
    return Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                     ('ohe', OneHotEncoder(handle_unknown='ignore'))])

def compute_eo(y_true, y_pred, s):
    g = np.unique(s)
    if len(g) != 2:
        return 0.0
    tpr, fpr = [], []
    for gi in g:
        pos = (s == gi) & (y_true == 1)
        neg = (s == gi) & (y_true == 0)
        tpr.append(y_pred[pos].mean() if pos.sum() > 0 else 0.0)
        fpr.append(y_pred[neg].mean() if neg.sum() > 0 else 0.0)
    return float(max(abs(tpr[0] - tpr[1]), abs(fpr[0] - fpr[1])))

def compute_dp(y_pred, s):
    g = np.unique(s)
    if len(g) != 2:
        return 0.0
    rates = [y_pred[s == gi].mean() for gi in g]
    return float(abs(rates[0] - rates[1]))

def compute_fairness_metrics(y_true, y_pred, sensitive_dict: dict) -> dict:
    metrics = {}
    yt = np.asarray(y_true)
    yp = np.asarray(y_pred)
    for name, values in sensitive_dict.items():
        sv = np.asarray(values).astype(int).flatten()
        if np.unique(sv).size != 2:
            metrics[f"{name}_eo"] = 0.0
            metrics[f"{name}_dp"] = 0.0
            continue
        try:
            eo = equalized_odds_difference(yt, yp, sensitive_features=sv)
            metrics[f"{name}_eo"] = 0.0 if np.isnan(eo) else abs(float(eo))
        except Exception:
            metrics[f"{name}_eo"] = compute_eo(yt, yp, sv)
        try:
            dp = demographic_parity_difference(yt, yp, sensitive_features=sv)
            metrics[f"{name}_dp"] = 0.0 if np.isnan(dp) else abs(float(dp))
        except Exception:
            metrics[f"{name}_dp"] = compute_dp(yp, sv)
    return metrics

def _sens_list_from_data(data: dict, dataset_key: str) -> List[np.ndarray]:
    cfg = DATASET_CONFIG[dataset_key]
    out = []
    for _, test_key in cfg["sens_attrs"]:
        if test_key in data:
            out.append(np.asarray(data[test_key]).flatten())
    return out

def _sens_dict_from_data(data: dict, dataset_key: str) -> dict:
    cfg = DATASET_CONFIG[dataset_key]
    out = {}
    for _, test_key in cfg["sens_attrs"]:
        if test_key in data:
            attr_name = test_key.replace("sens_", "").replace("_test", "")
            out[attr_name] = np.asarray(data[test_key]).flatten()
    return out

def _sens_train_list(data: dict, dataset_key: str) -> List[np.ndarray]:
    cfg = DATASET_CONFIG[dataset_key]
    out = []
    for train_key, _ in cfg["sens_attrs"]:
        if train_key in data:
            out.append(np.asarray(data[train_key]).flatten())
    return out

def _sens_train_dict(data: dict, dataset_key: str) -> dict:
    cfg = DATASET_CONFIG[dataset_key]
    out = {}
    for train_key, _ in cfg["sens_attrs"]:
        if train_key in data:
            attr_name = train_key.replace("sens_", "").replace("_train", "")
            out[attr_name] = np.asarray(data[train_key]).flatten()
    return out

print("Utility helpers defined.")


# ─────────────────────────────────────────────────────────────
# Preprocessing functions (all 6, unchanged from v2)
# ─────────────────────────────────────────────────────────────

def preprocess_adult_for_fair_bbn(
        csv_path='/kaggle/input/datasets/maansirao/all-datasets/adult.csv',
        seed=SEED, use_cache=True):
    cache_file = os.path.join(CACHE_DIR, 'preproc_adult.pkl')
    if use_cache and os.path.exists(cache_file):
        tqdm.write("  [cache] Adult data loaded.")
        return joblib.load(cache_file)

    col_names = ['age','workclass','fnlwgt','education','education-num',
                 'marital-status','occupation','relationship','race','sex',
                 'capital-gain','capital-loss','hours-per-week','native-country','income']
    df = None
    for sep in [', ', ',']:
        try:
            peek = pd.read_csv(csv_path, sep=sep, nrows=1, header=0)
            if peek.shape[1] == 15:
                first_val = str(peek.iloc[0, 0]).strip()
                if first_val.lstrip('-').isdigit():
                    df = pd.read_csv(csv_path, sep=sep, names=col_names, header=None)
                else:
                    df = pd.read_csv(csv_path, sep=sep, header=0)
                    df.columns = col_names
                break
        except Exception:
            continue
    if df is None:
        df = pd.read_csv(csv_path)
        if df.shape[1] == 15:
            df.columns = col_names
        else:
            raise ValueError(f"Cannot parse Adult CSV: shape {df.shape}")
    df.columns = col_names

    for col in df.select_dtypes(include='object').columns:
        df[col] = df[col].astype(str).str.strip()
    df = df[~df.isin(['?']).any(axis=1)].reset_index(drop=True)
    df = df.drop(columns=['fnlwgt'])

    income_col = df['income'].astype(str).str.strip()
    df['income_label'] = income_col.str.contains('>50K', na=False).astype(int)
    if df['income_label'].sum() == 0:
        df['income_label'] = pd.to_numeric(df['income'], errors='coerce').fillna(0).astype(int)

    df['sex_binary']  = (df['sex'].astype(str).str.strip() == 'Male').astype(int)
    df['race_binary'] = (df['race'].astype(str).str.strip() == 'White').astype(int)

    numeric_cols    = ['age','education-num','capital-gain','capital-loss','hours-per-week']
    categorical_cols = ['workclass','education','marital-status','occupation',
                        'relationship','native-country']
    for col in numeric_cols:
        df[col] = clean_numeric_column(df[col])
    for col in ['capital-gain','capital-loss']:
        df[col] = df[col].clip(upper=df[col].quantile(0.99))

    preproc = ColumnTransformer([('num', make_num_pipeline(), numeric_cols),
                                 ('cat', make_cat_pipeline(), categorical_cols)])

    bbn_df = df.copy()
    for col in numeric_cols:
        bbn_df[col] = safe_qcut(bbn_df[col], 5)
    for col in categorical_cols + ['sex', 'race']:
        bbn_df[col] = bbn_df[col].astype('category').cat.codes.astype(int)

    X = df.drop(columns=['income','income_label','sex_binary','race_binary','sex','race'])
    y = df['income_label'].values
    for col in X.select_dtypes(include=[np.number]).columns:
        X[col] = clean_numeric_column(X[col])

    (X_tr, X_te, y_tr, y_te, ss_tr, ss_te, sr_tr, sr_te) = train_test_split(
        X, y, df['sex_binary'].values, df['race_binary'].values,
        test_size=0.2, stratify=y, random_state=seed)

    X_train_nn = to_dense(preproc.fit_transform(X_tr))
    X_test_nn  = to_dense(preproc.transform(X_te))
    bbn_train  = bbn_df.loc[X_tr.index].reset_index(drop=True)
    bbn_test   = bbn_df.loc[X_te.index].reset_index(drop=True)

    result = dict(X_train_nn=X_train_nn, X_test_nn=X_test_nn,
                  bbn_train_df=bbn_train, bbn_test_df=bbn_test,
                  y_train=y_tr, y_test=y_te,
                  sens_sex_train=ss_tr, sens_sex_test=ss_te,
                  sens_race_train=sr_tr, sens_race_test=sr_te)
    if use_cache: joblib.dump(result, cache_file)
    return result


def preprocess_compas_for_fair_bbn(
        csv_path='/kaggle/input/datasets/maansirao/all-datasets/compas-scores-two-years.csv',
        seed=SEED, use_cache=True):
    cache_file = os.path.join(CACHE_DIR, 'preproc_compas.pkl')
    if use_cache and os.path.exists(cache_file):
        tqdm.write("  [cache] COMPAS data loaded.")
        return joblib.load(cache_file)

    df = pd.read_csv(csv_path)
    df = df.loc[
        (df['days_b_screening_arrest'] <= 30) & (df['days_b_screening_arrest'] >= -30) &
        (df['is_recid'] != -1) & (df['c_charge_degree'] != 'O') & (df['score_text'] != 'N/A'),
        ['age','c_charge_degree','race','age_cat','score_text','sex','priors_count',
         'days_b_screening_arrest','decile_score','juv_other_count','juv_misd_count',
         'juv_fel_count','c_charge_desc','is_recid','two_year_recid','c_jail_in','c_jail_out']
    ].reset_index(drop=True)

    df['race_label'] = df['race'].map({"African-American":0,"Caucasian":1,"Hispanic":2,
                                        "Other":3,"Asian":4,"Native American":5})
    df['sex_label']  = df['sex'].map({"Male":0,"Female":1})
    df['c_jail_in']  = pd.to_datetime(df['c_jail_in'])
    df['c_jail_out'] = pd.to_datetime(df['c_jail_out'])
    df['jail_time']  = (df['c_jail_out'] - df['c_jail_in']).dt.days.fillna(0)
    df = df.drop(columns=['c_jail_in','c_jail_out'])
    df['race_binary'] = (df['race_label'] == 0).astype(int)
    df['sex_binary']  = df['sex_label']

    numeric_cols    = ['age','priors_count','days_b_screening_arrest','decile_score',
                       'jail_time','juv_other_count','juv_misd_count','juv_fel_count']
    categorical_cols = ['c_charge_degree','race','age_cat','score_text','sex','c_charge_desc']
    for col in numeric_cols:
        df[col] = clean_numeric_column(df[col])

    preproc = ColumnTransformer([('num', make_num_pipeline(), numeric_cols),
                                 ('cat', make_cat_pipeline(), categorical_cols)])

    bbn_df = df.copy()
    for col in numeric_cols:
        bbn_df[col] = safe_qcut(bbn_df[col], 5)
    for col in categorical_cols + ['race','sex']:
        bbn_df[col] = bbn_df[col].astype('category').cat.codes.astype(int)

    X = df.drop(columns=['is_recid','two_year_recid','race_label','sex_label',
                          'race_binary','sex_binary'])
    y = df['two_year_recid'].values
    (X_tr, X_te, y_tr, y_te, sr_tr, sr_te, ss_tr, ss_te) = train_test_split(
        X, y, df['race_binary'], df['sex_binary'],
        test_size=0.2, stratify=y, random_state=seed)

    X_train_nn = to_dense(preproc.fit_transform(X_tr))
    X_test_nn  = to_dense(preproc.transform(X_te))
    bbn_train  = bbn_df.loc[X_tr.index].reset_index(drop=True)
    bbn_test   = bbn_df.loc[X_te.index].reset_index(drop=True)

    result = dict(X_train_nn=X_train_nn, X_test_nn=X_test_nn,
                  bbn_train_df=bbn_train, bbn_test_df=bbn_test,
                  y_train=y_tr, y_test=y_te,
                  sens_race_train=sr_tr.reset_index(drop=True),
                  sens_race_test=sr_te.reset_index(drop=True),
                  sens_sex_train=ss_tr.reset_index(drop=True),
                  sens_sex_test=ss_te.reset_index(drop=True))
    if use_cache: joblib.dump(result, cache_file)
    return result


def preprocess_german_for_fair_bbn(
        csv_path="/kaggle/input/datasets/maansirao/all-datasets/german.data",
        seed=SEED, use_cache=True):
    cache_file = os.path.join(CACHE_DIR, 'preproc_german.pkl')
    if use_cache and os.path.exists(cache_file):
        tqdm.write("  [cache] German data loaded.")
        return joblib.load(cache_file)

    col_names = ["status","duration","credit_history","purpose","amount","savings",
                 "employment","installment_rate","personal_status_sex","other_debtors",
                 "residence","property","age","other_installment_plans","housing",
                 "number_credits","job","people_liable","telephone","foreign_worker","credit"]
    df = pd.read_csv(csv_path, sep=' ', names=col_names)
    sex_map = {'A91':'male','A92':'male','A93':'male','A94':'female','A95':'female'}
    df['sex'] = df['personal_status_sex'].map(sex_map)
    df['sex_label']            = df['sex'].map({'male':0,'female':1})
    df['age_label']            = (df['age'] >= 25).astype(int)
    df['foreign_worker_label'] = df['foreign_worker'].map({'A201':1,'A202':0})
    df['credit_label']         = df['credit'].map({1:1, 2:0})
    df = df.drop(columns=['personal_status_sex','sex','age','foreign_worker','credit'])

    numerical_cols  = ["duration","amount","installment_rate","residence",
                       "number_credits","people_liable"]
    categorical_cols = [c for c in df.columns if c not in
                        numerical_cols + ['sex_label','age_label','foreign_worker_label','credit_label']]
    for col in numerical_cols:
        df[col] = clean_numeric_column(df[col])

    preproc = ColumnTransformer([('num', make_num_pipeline(), numerical_cols),
                                 ('cat', make_cat_pipeline(), categorical_cols)])

    bbn_df = df.copy()
    for col in numerical_cols:
        bbn_df[col] = safe_qcut(bbn_df[col], 5)
    for col in categorical_cols + ['sex_label','age_label','foreign_worker_label']:
        bbn_df[col] = bbn_df[col].astype('category').cat.codes.astype(int)

    X = df.drop(columns=['credit_label'])
    y = df['credit_label'].values
    (X_tr, X_te, y_tr, y_te, sa_tr, sa_te, ss_tr, ss_te) = train_test_split(
        X, y, df['age_label'].values, df['sex_label'].values,
        test_size=0.2, stratify=y, random_state=seed)

    X_train_nn = to_dense(preproc.fit_transform(X_tr))
    X_test_nn  = to_dense(preproc.transform(X_te))
    bbn_train  = bbn_df.loc[X_tr.index].reset_index(drop=True)
    bbn_test   = bbn_df.loc[X_te.index].reset_index(drop=True)

    result = dict(X_train_nn=X_train_nn, X_test_nn=X_test_nn,
                  bbn_train_df=bbn_train, bbn_test_df=bbn_test,
                  y_train=y_tr, y_test=y_te,
                  sens_age_train=sa_tr, sens_age_test=sa_te,
                  sens_sex_train=ss_tr, sens_sex_test=ss_te)
    if use_cache: joblib.dump(result, cache_file)
    return result


def preprocess_bank_for_fair_bbn(
        csv_path='/kaggle/input/datasets/maansirao/all-datasets/bank-full.csv',
        seed=SEED, use_cache=True):
    cache_file = os.path.join(CACHE_DIR, 'preproc_bank.pkl')
    if use_cache and os.path.exists(cache_file):
        tqdm.write("  [cache] Bank data loaded.")
        return joblib.load(cache_file)

    try:
        df = pd.read_csv(csv_path, sep=';')
    except Exception:
        df = pd.read_csv(csv_path, sep=',')
    df = df.apply(lambda x: x.str.lower() if x.dtype == 'object' else x)
    target_col = ('y' if 'y' in df.columns else 'deposit' if 'deposit' in df.columns else df.columns[-1])
    df = df[~df.isin(['unknown']).any(axis=1)].reset_index(drop=True)
    df['y'] = np.where(df[target_col].isin(['yes','y','true','1']), 1, 0)
    df['marital_binary'] = (df['marital'] == df['marital'].value_counts().idxmax()).astype(int)
    df['job_binary']     = (df['job']     == df['job'].value_counts().idxmax()).astype(int)

    categorical_cols = [c for c in ['job','education','default','housing','loan',
                                    'contact','month','poutcome'] if c in df.columns]
    numeric_cols     = [c for c in ['age','balance','day','duration','campaign','pdays','previous']
                        if c in df.columns]
    for col in numeric_cols:
        df[col] = clean_numeric_column(df[col])
    for col in ['balance','duration','pdays','previous']:
        if col in df.columns:
            df[col] = df[col].clip(upper=df[col].quantile(0.99))

    preproc = ColumnTransformer([('num', make_num_pipeline(), numeric_cols),
                                 ('cat', make_cat_pipeline(), categorical_cols)])

    bbn_df = df.copy()
    for col in numeric_cols:
        bbn_df[col] = safe_qcut(bbn_df[col], 5)
    for col in categorical_cols + ['marital','job']:
        bbn_df[col] = bbn_df[col].astype('category').cat.codes.astype(int)

    X = df.drop(columns=['y','marital_binary','job_binary'])
    y = df['y'].values
    (X_tr, X_te, y_tr, y_te, sm_tr, sm_te, sj_tr, sj_te) = train_test_split(
        X, y, df['marital_binary'], df['job_binary'],
        test_size=0.2, stratify=y, random_state=seed)

    X_train_nn = to_dense(preproc.fit_transform(X_tr))
    X_test_nn  = to_dense(preproc.transform(X_te))
    bbn_train  = bbn_df.loc[X_tr.index].reset_index(drop=True)
    bbn_test   = bbn_df.loc[X_te.index].reset_index(drop=True)

    result = dict(X_train_nn=X_train_nn, X_test_nn=X_test_nn,
                  bbn_train_df=bbn_train, bbn_test_df=bbn_test,
                  y_train=y_tr, y_test=y_te,
                  sens_marital_train=sm_tr.reset_index(drop=True),
                  sens_marital_test=sm_te.reset_index(drop=True),
                  sens_job_train=sj_tr.reset_index(drop=True),
                  sens_job_test=sj_te.reset_index(drop=True))
    if use_cache: joblib.dump(result, cache_file)
    return result


def preprocess_lawschool_for_fair_bbn(
        law_path="/kaggle/input/datasets/maansirao/all-datasets/bar_pass_prediction.csv",
        use_cache=True):
    cache_file = os.path.join(CACHE_DIR, 'preproc_lawschool.pkl')
    if use_cache and os.path.exists(cache_file):
        tqdm.write("  [cache] Law School data loaded.")
        return joblib.load(cache_file)

    law_df = pd.read_csv(law_path)
    law_df.columns = [c.strip().lower() for c in law_df.columns]
    law_df = law_df.dropna(subset=['pass_bar','race','sex'])
    for col in law_df.select_dtypes(include=[np.number]).columns:
        law_df[col] = clean_numeric_column(law_df[col])

    most_common_race      = law_df['race'].value_counts().idxmax()
    law_df['race_binary'] = (law_df['race'] == most_common_race).astype(int)
    gender_map            = {v: i for i, v in enumerate(sorted(law_df['sex'].unique()))}
    law_df['gender_binary'] = law_df['sex'].map(gender_map)

    numeric_cols    = [c for c in ['lsat','ugpa','zfygpa','zgpa','fam_inc','age','gpa']
                       if c in law_df.columns and law_df[c].nunique() > 1]
    categorical_cols = [c for c in ['tier','cluster','fulltime','parttime']
                        if c in law_df.columns]

    preproc = ColumnTransformer([('num', make_num_pipeline(), numeric_cols),
                                 ('cat', make_cat_pipeline(), categorical_cols)])

    bbn_df = law_df.copy()
    for col in numeric_cols:
        bbn_df[col] = safe_qcut(law_df[col], 5)
    for col in categorical_cols + ['race','sex']:
        bbn_df[col] = bbn_df[col].astype('category').cat.codes.astype(int)

    X = law_df[numeric_cols + categorical_cols + ['race','sex']]
    y = law_df['pass_bar'].values
    (X_tr, X_te, y_tr, y_te, sr_tr, sr_te, sg_tr, sg_te) = train_test_split(
        X, y, law_df['race_binary'], law_df['gender_binary'],
        test_size=0.2, stratify=y, random_state=SEED)

    X_train_nn = to_dense(preproc.fit_transform(X_tr))
    X_test_nn  = to_dense(preproc.transform(X_te))
    bbn_train  = bbn_df.loc[X_tr.index].reset_index(drop=True)
    bbn_test   = bbn_df.loc[X_te.index].reset_index(drop=True)

    result = dict(X_train_nn=X_train_nn, X_test_nn=X_test_nn,
                  bbn_train_df=bbn_train, bbn_test_df=bbn_test,
                  y_train=y_tr, y_test=y_te,
                  sens_race_train=sr_tr.reset_index(drop=True),
                  sens_race_test=sr_te.reset_index(drop=True),
                  sens_gender_train=sg_tr.reset_index(drop=True),
                  sens_gender_test=sg_te.reset_index(drop=True))
    if use_cache: joblib.dump(result, cache_file)
    return result


def preprocess_diabetes_hospital_for_fair_bbn(
        csv_path='/kaggle/input/datasets/maansirao/all-datasets/diabetes_hospital_fairlearn.csv',
        seed=SEED, use_cache=True):
    cache_file = os.path.join(CACHE_DIR, 'preproc_hospital.pkl')
    if use_cache and os.path.exists(cache_file):
        tqdm.write("  [cache] Hospital data loaded.")
        return joblib.load(cache_file)

    df = pd.read_csv(csv_path)
    df = df.drop(columns=[c for c in ["max_glu_serum","A1Cresult","readmitted.1"]
                           if c in df.columns])
    df = df[~df.isin(['Missing']).any(axis=1)]
    df = df.dropna(subset=['race','gender']).reset_index(drop=True)

    age_map = {"'0-10'":5,"'10-20'":15,"'20-30'":25,"'30-40'":35,"'40-50'":45,
               "'50-60'":55,"'60-70'":65,"'70-80'":75,"'80-90'":85,"'90-100'":95,
               "'30 years or younger'":20,"'30-60 years'":45,"'Over 60 years'":70}
    df['age'] = df['age'].replace(age_map).astype(float)
    df['readmit_binary'] = df['readmit_binary'].astype(int)

    most_common_race    = df['race'].value_counts().idxmax()
    df['race_binary']   = (df['race'] == most_common_race).astype(int)
    df['gender_binary'] = df['gender'].map({'Male':0,'Female':1}).fillna(0).astype(int)

    categorical_cols = ['discharge_disposition_id','admission_source_id','medical_specialty',
                        'primary_diagnosis','insulin','change','diabetesMed']
    numeric_cols     = ['age','time_in_hospital','num_lab_procedures','num_procedures',
                        'num_medications','number_diagnoses','had_emergency',
                        'had_inpatient_days','had_outpatient_days','medicare','medicaid']
    for col in numeric_cols:
        df[col] = clean_numeric_column(df[col])

    preproc = ColumnTransformer([('num', make_num_pipeline(), numeric_cols),
                                 ('cat', make_cat_pipeline(), categorical_cols)])

    bbn_df = df.copy()
    for col in numeric_cols:
        bbn_df[col] = safe_qcut(bbn_df[col], 5)
    for col in categorical_cols + ['race','gender']:
        bbn_df[col] = bbn_df[col].astype('category').cat.codes.astype(int)

    X = df.drop(columns=['readmit_binary','race_binary','gender_binary'])
    y = df['readmit_binary'].values
    (X_tr, X_te, y_tr, y_te, sr_tr, sr_te, sg_tr, sg_te) = train_test_split(
        X, y, df['race_binary'], df['gender_binary'],
        test_size=0.2, stratify=y, random_state=seed)

    X_train_nn = to_dense(preproc.fit_transform(X_tr))
    X_test_nn  = to_dense(preproc.transform(X_te))
    bbn_train  = bbn_df.loc[X_tr.index].reset_index(drop=True)
    bbn_test   = bbn_df.loc[X_te.index].reset_index(drop=True)

    result = dict(X_train_nn=X_train_nn, X_test_nn=X_test_nn,
                  bbn_train_df=bbn_train, bbn_test_df=bbn_test,
                  y_train=y_tr, y_test=y_te,
                  sens_race_train=sr_tr.reset_index(drop=True),
                  sens_race_test=sr_te.reset_index(drop=True),
                  sens_sex_train=sg_tr.reset_index(drop=True),
                  sens_sex_test=sg_te.reset_index(drop=True))
    if use_cache: joblib.dump(result, cache_file)
    return result

print("All 6 preprocessing functions defined.")


class BBNFairnessModel:
    def __init__(self, target_col='label', sens_cols=None, max_rows=BBN_MAX_ROWS):
        self.target_col = target_col
        self.sens_cols  = sens_cols or []
        self.max_rows   = max_rows
        self.model      = None
        self.infer      = None
        self.fitted     = False

    def fit(self, bbn_df: pd.DataFrame, y_train: np.ndarray):
        df = bbn_df.copy()
        df[self.target_col] = y_train.astype(int)
        if len(df) > self.max_rows:
            df = df.sample(self.max_rows, random_state=SEED)
        for col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int).clip(0, 9)
        edges = [(col, self.target_col) for col in self.sens_cols if col in df.columns]
        if not edges:
            self.fitted = False
            return self
        try:
            cols_needed = list({s for s in self.sens_cols if s in df.columns} | {self.target_col})
            self.model = DiscreteBayesianNetwork(edges)
            self.model.fit(df[cols_needed], estimator=BayesianEstimator,
                           prior_type='BDeu', equivalent_sample_size=5)
            self.infer  = VariableElimination(self.model)
            self.fitted = True
        except Exception as e:
            self.fitted = False
        return self

    def predict_proba(self, bbn_df: pd.DataFrame) -> np.ndarray:
        if not self.fitted or self.infer is None:
            return np.full(len(bbn_df), 0.5)
        probs = np.full(len(bbn_df), 0.5)
        for i, row in bbn_df.iterrows():
            try:
                evidence = {col: int(pd.to_numeric(row[col], errors='coerce') or 0)
                            for col in self.sens_cols
                            if col in bbn_df.columns and not pd.isna(row[col])}
                if not evidence:
                    continue
                q = self.infer.query([self.target_col], evidence=evidence, show_progress=False)
                vals = q.values
                probs[i] = float(vals[1]) if len(vals) > 1 else float(vals[0])
            except Exception:
                pass
        return probs

    def predict_fair_targets(self, bbn_df, y_true, fairness_dir=True):
        if not self.fitted:
            return y_true.astype(float)
        bbn_probs = self.predict_proba(bbn_df)
        global_pos_rate = y_true.mean()
        if fairness_dir:
            correction = global_pos_rate - bbn_probs
            soft = np.clip(y_true.astype(float) + 0.3 * correction, 0.05, 0.95)
        else:
            soft = np.full(len(y_true), global_pos_rate)
            soft = np.clip(0.7 * y_true.astype(float) + 0.3 * soft, 0.05, 0.95)
        return soft


class FairnessEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, fairness_dim, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, fairness_dim),
            nn.BatchNorm1d(fairness_dim),
            nn.ReLU(),
        )

    def forward(self, x):
        return self.net(x)


class Classifier(nn.Module):
    def __init__(self, fairness_dim, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(fairness_dim, fairness_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(fairness_dim // 2, 1),
        )

    def forward(self, z):
        return self.net(z).squeeze(-1)


class SensitivityAdversary(nn.Module):
    def __init__(self, fairness_dim, n_sensitive, dropout):
        super().__init__()
        self.heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(fairness_dim, fairness_dim // 2),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(fairness_dim // 2, 1),
            ) for _ in range(n_sensitive)
        ])

    def forward(self, z):
        return [h(z).squeeze(-1) for h in self.heads]


print("Neural network modules defined.")


# ─────────────────────────────────────────────────────────────
# NOVEL COMPONENT 4: EO-Primary / DP-Primary Threshold Sweep
# ─────────────────────────────────────────────────────────────
# Mode-aware: in 'eo_only', primary objective is EO; DP is
# reported but not optimised. In 'dp_only', primary is DP.
# In 'both', original dual-objective scoring applies.
#
# Key new constraint: sweep_tau_min/max limits per dataset
# prevent the extreme thresholds (0.02 / 0.958) that caused
# the accuracy drops in v2. If the constrained sweep fails,
# we fall back to the full range.
# ─────────────────────────────────────────────────────────────

def _primary_score(acc, eo, dp, baseline_acc, max_drop, mode='eo_only'):
    if acc < baseline_acc - max_drop:
        return np.inf
    if mode == 'eo_only':
        fl = floor2(eo)
        if fl == 0.0:
            return max(0.0, baseline_acc - acc)
        return fl * 100 + eo * 0.3 + max(0.0, baseline_acc - acc) * 0.1
    elif mode == 'dp_only':
        fl = floor2(dp)
        if fl == 0.0:
            return max(0.0, baseline_acc - acc)
        return fl * 100 + dp * 0.3 + max(0.0, baseline_acc - acc) * 0.1
    else:  # 'both'
        fl_eo = floor2(eo); fl_dp = floor2(dp)
        if fl_eo == 0.0:
            return max(0.0, baseline_acc - acc) + fl_dp * 0.5
        return fl_eo * 100 + fl_dp * 5 + (eo + dp) * 0.3


def joint_threshold_sweep(
        y_true:       np.ndarray,
        y_proba:      np.ndarray,
        s_list:       List[np.ndarray],
        hp:           DatasetHParams,
        baseline_acc: float,
        mode:         str = 'eo_only',
) -> Tuple[np.ndarray, float]:
    y_true  = np.asarray(y_true)
    y_proba = np.asarray(y_proba, dtype=float)
    max_drop = hp.acc_drop_fallback
    n = len(y_proba)

    def _metrics(yp):
        acc = accuracy_score(y_true, yp)
        eo  = max((compute_eo(y_true, yp, s) for s in s_list), default=0.0)
        dp  = max((compute_dp(yp, s) for s in s_list), default=0.0)
        return acc, eo, dp

    def _score(acc, eo, dp):
        return _primary_score(acc, eo, dp, baseline_acc, max_drop, mode)

    # ── Step 1: constrained global sweep (new) ─────────────────
    thresholds_constrained = np.linspace(hp.sweep_tau_min, hp.sweep_tau_max, 401)
    best_pred  = (y_proba >= hp.tau).astype(int)
    best_score = np.inf
    best_tau   = hp.tau

    for tau in thresholds_constrained:
        yp  = (y_proba >= tau).astype(int)
        acc, eo, dp = _metrics(yp)
        sc  = _score(acc, eo, dp)
        if sc < best_score:
            best_score = sc
            best_pred  = yp.copy()
            best_tau   = tau

    # ── Step 1b: fallback to full range if constrained sweep fails
    acc_b, eo_b, dp_b = _metrics(best_pred)
    primary_met = (floor2(eo_b) == 0.0 if mode in ('eo_only', 'both')
                   else floor2(dp_b) == 0.0)
    if not primary_met:
        thresholds_full = np.linspace(0.02, 0.98, 961)
        for tau in thresholds_full:
            yp  = (y_proba >= tau).astype(int)
            acc, eo, dp = _metrics(yp)
            sc  = _score(acc, eo, dp)
            if sc < best_score:
                best_score = sc
                best_pred  = yp.copy()
                best_tau   = tau

    acc_b, eo_b, dp_b = _metrics(best_pred)
    primary_met = (floor2(eo_b) == 0.0 if mode in ('eo_only', 'both')
                   else floor2(dp_b) == 0.0)
    if primary_met:
        return best_pred, best_tau

    if not hp.use_group_threshold:
        return best_pred, best_tau

    # ── Step 2: per-group threshold sweep ─────────────────────
    tau_range = np.linspace(hp.sweep_tau_min, hp.sweep_tau_max, 181)

    for s_arr in [np.asarray(s) for s in s_list]:
        mask1 = (s_arr == 1)
        mask0 = ~mask1

        p1 = y_proba[mask1]
        p0 = y_proba[mask0]

        preds_g1 = (p1[None, :] >= tau_range[:, None])
        preds_g0 = (p0[None, :] >= tau_range[:, None])

        best_gp = best_pred.copy()
        best_gs = best_score

        for i0, t0 in enumerate(tau_range):
            p0_pred = preds_g0[i0].astype(np.int8)
            for i1, t1 in enumerate(tau_range):
                p1_pred = preds_g1[i1].astype(np.int8)

                yp = np.empty(n, dtype=np.int8)
                yp[mask1] = p1_pred
                yp[mask0] = p0_pred

                acc = (yp == y_true).mean()
                if acc < baseline_acc - max_drop:
                    continue

                eo = max((compute_eo(y_true, yp.astype(int), s) for s in s_list), default=0.0)
                dp = max((compute_dp(yp.astype(int), s) for s in s_list), default=0.0)
                sc = _score(acc, eo, dp)

                if sc < best_gs:
                    best_gs = sc
                    best_gp = yp.astype(int).copy()

        if best_gs < best_score:
            best_score = best_gs
            best_pred  = best_gp.copy()

        acc_b, eo_b, dp_b = _metrics(best_pred)
        primary_met = (floor2(eo_b) == 0.0 if mode in ('eo_only', 'both')
                       else floor2(dp_b) == 0.0)
        if primary_met:
            return best_pred, best_tau

    # ── Step 3: fine-grained search near best_tau ────────────
    fine_taus = np.arange(
        max(0.01, best_tau - 0.08),
        min(0.99, best_tau + 0.08),
        0.001
    )
    for tau in fine_taus:
        yp  = (y_proba >= tau).astype(int)
        acc, eo, dp = _metrics(yp)
        sc  = _score(acc, eo, dp)
        if sc < best_score:
            best_score = sc
            best_pred  = yp.copy()
            best_tau   = tau

    # ── Step 4: borderline-flip greedy search ────────────────
    border_order = np.argsort(np.abs(y_proba - best_tau))
    flip_budget  = min(300, int(n * 0.05))
    current_pred = best_pred.copy()
    for idx in border_order[:flip_budget]:
        trial = current_pred.copy()
        trial[idx] ^= 1
        acc, eo, dp = _metrics(trial)
        sc = _score(acc, eo, dp)
        if sc < best_score:
            best_score   = sc
            best_pred    = trial.copy()
            current_pred = trial.copy()
            primary_met  = (floor2(eo) == 0.0 if mode in ('eo_only', 'both')
                            else floor2(dp) == 0.0)
            if primary_met:
                break

    return best_pred, best_tau


def calibrate_proba(y_proba_train, y_train, y_proba_test):
    iso = IsotonicRegression(out_of_bounds='clip')
    iso.fit(y_proba_train, y_train)
    return iso.transform(y_proba_test), iso.transform(y_proba_train)


print("Threshold sweep defined.")


def _tensor(arr, dtype=torch.float32):
    return torch.tensor(np.asarray(arr), dtype=dtype).to(DEVICE)

def _get_proba(encoder, classifier, X_tensor, batch=2048):
    encoder.eval(); classifier.eval()
    parts = []
    with torch.no_grad():
        for i in range(0, len(X_tensor), batch):
            z = encoder(X_tensor[i:i+batch])
            parts.append(torch.sigmoid(classifier(z)).cpu().numpy())
    return np.concatenate(parts)


# ─────────────────────────────────────────────────────────────
# DIAGNOSTIC SCRIPT
# ─────────────────────────────────────────────────────────────
# Call run_diagnostics(data, dataset_key, results) after training
# to get a breakdown of where accuracy is being lost.
# ─────────────────────────────────────────────────────────────

def run_diagnostics(
    dataset_key:       str,
    baseline_acc:      float,
    diag_checkpoints:  List[dict],   # list of dicts, each from _diag_checkpoint()
    final_result:      dict,
):
    """
    Prints a diagnostic table showing accuracy and fairness at each
    pipeline stage. Identifies where accuracy is being lost.

    diag_checkpoints entries:
        {'stage': str, 'acc': float, 'eo': float, 'dp': float,
         'tau': float or None, 'note': str}
    """
    print(f"\n  {'─'*60}")
    print(f"  DIAGNOSTICS: {dataset_key.upper()}")
    print(f"  {'─'*60}")
    print(f"  {'Stage':<28} {'Acc':>7} {'ΔAcc':>7} {'EO':>7} {'EO_fl':>6} {'DP':>7} {'tau':>6}")
    print(f"  {'─'*60}")

    prev_acc = baseline_acc
    for ck in diag_checkpoints:
        stage  = ck.get('stage', '?')
        acc    = ck.get('acc', 0.0)
        eo     = ck.get('eo', 0.0)
        dp     = ck.get('dp', 0.0)
        tau    = ck.get('tau', None)
        note   = ck.get('note', '')
        d_acc  = acc - prev_acc
        tau_s  = f"{tau:.3f}" if tau is not None else "  —  "
        # Flag stages where >2% accuracy is lost
        flag = " ◄ LOSS" if d_acc < -0.02 else ""
        print(f"  {stage:<28} {acc:7.4f} {d_acc:+7.4f} {eo:7.4f} "
              f"{floor2(eo):6.2f} {dp:7.4f} {tau_s:>6}{flag}")
        if note:
            print(f"  {'':>28} note: {note}")
        prev_acc = acc

    # Summary
    eo_final = max(v for k, v in final_result.items() if k.endswith('_eo'))
    dp_final = max(v for k, v in final_result.items() if k.endswith('_dp'))
    total_drop = baseline_acc - final_result.get('acc', 0.0)
    print(f"  {'─'*60}")
    print(f"  Total accuracy drop: {total_drop:+.4f}")
    print(f"  Final EO_fl={floor2(eo_final):.2f}  DP_fl={floor2(dp_final):.2f}")
    print(f"  Mode: {PIPELINE_MODE}")
    print(f"  {'─'*60}\n")


def train_model(
        data:                  dict,
        dataset_key:           str,
        hp:                    DatasetHParams,
        original_baseline_acc: float,
        verbose:               bool = True,
        mode:                  str  = 'eo_only',
) -> dict:
    set_seed(SEED)
    y_train = np.asarray(data['y_train'])
    y_test  = np.asarray(data['y_test'])
    X_tr    = _tensor(data['X_train_nn'])
    X_te    = _tensor(data['X_test_nn'])

    s_train = _sens_train_list(data, dataset_key)
    s_test  = _sens_list_from_data(data, dataset_key)
    s_dict  = _sens_dict_from_data(data, dataset_key)
    n_sens  = len(s_train)

    input_dim = X_tr.shape[1]

    encoder    = FairnessEncoder(input_dim, hp.hidden_dim, hp.fairness_dim, hp.dropout).to(DEVICE)
    classifier = Classifier(hp.fairness_dim, hp.dropout).to(DEVICE)
    adversary  = SensitivityAdversary(hp.fairness_dim, n_sens, hp.dropout).to(DEVICE)

    pos_weight = torch.tensor([(y_train == 0).sum() / max(1, (y_train == 1).sum())],
                               dtype=torch.float32).to(DEVICE)
    bce     = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    bce_unw = nn.BCEWithLogitsLoss()

    yt_tr        = _tensor(y_train)
    s_tensors_tr = [_tensor(s.astype(float)) for s in s_train]

    ds_tr = TensorDataset(X_tr, yt_tr, *s_tensors_tr)
    dl_tr = DataLoader(ds_tr, batch_size=hp.batch_size, shuffle=True, drop_last=False)

    bbn_prior_train   = np.full(len(y_train), 0.5)
    bbn_fair_targets  = y_train.astype(float)
    bbn_model         = None

    if hp.use_bbn:
        try:
            cfg       = DATASET_CONFIG[dataset_key]
            sens_cols = [tk.replace("sens_","").replace("_train","")
                         for tk, _ in cfg["sens_attrs"] if tk in data]
            bbn_model = BBNFairnessModel(target_col='label', sens_cols=sens_cols)
            bbn_model.fit(data['bbn_train_df'], y_train)
            if bbn_model.fitted:
                bbn_prior_train  = bbn_model.predict_proba(data['bbn_train_df'])
                bbn_fair_targets = bbn_eo_guided_targets(
                    y_train, s_train, bbn_prior_train, alpha=0.25)
                if verbose:
                    print(f"  BBN prior fitted. mean={bbn_prior_train.mean():.3f} "
                          f"eo_targets_mean={bbn_fair_targets.mean():.3f}")
        except Exception as e:
            if verbose:
                print(f"  BBN skipped: {e}")

    bbn_prior_t       = _tensor(bbn_prior_train)
    bbn_fair_target_t = _tensor(bbn_fair_targets)

    # ── Stage 1: standard classification + BBN priors ────────
    opt1   = optim.Adam(list(encoder.parameters()) + list(classifier.parameters()),
                        lr=hp.lr, weight_decay=1e-5)
    sched1 = optim.lr_scheduler.CosineAnnealingLR(opt1, T_max=hp.encoder_epochs, eta_min=1e-6)
    best_val_loss  = np.inf
    patience_cnt   = 0
    best_enc_state = copy.deepcopy(encoder.state_dict())
    best_cls_state = copy.deepcopy(classifier.state_dict())

    for epoch in range(hp.encoder_epochs):
        encoder.train(); classifier.train()
        for batch in dl_tr:
            xb = batch[0]; yb = batch[1]
            batch_idx = torch.randint(0, len(bbn_prior_t), (len(xb),))
            z     = encoder(xb)
            logit = classifier(z)
            prob  = torch.sigmoid(logit)
            loss_cls = bce(logit, yb)
            loss_bbn_prior = torch.tensor(0.0, device=DEVICE)
            loss_bbn_fair  = torch.tensor(0.0, device=DEVICE)
            if hp.use_bbn and hp.bbn_prior_weight > 0 and bbn_model is not None and bbn_model.fitted:
                loss_bbn_prior = F.binary_cross_entropy(
                    prob, bbn_prior_t[batch_idx].clamp(0.05, 0.95))
                loss_bbn_fair = F.binary_cross_entropy(
                    prob, bbn_fair_target_t[batch_idx].clamp(0.05, 0.95))
            loss = (loss_cls
                    + hp.bbn_prior_weight * loss_bbn_prior
                    + hp.bbn_adv_weight   * loss_bbn_fair)
            opt1.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(
                list(encoder.parameters()) + list(classifier.parameters()), 1.0)
            opt1.step()
        sched1.step()

        with torch.no_grad():
            encoder.eval(); classifier.eval()
            z_te     = encoder(X_te)
            val_loss = bce(classifier(z_te), _tensor(y_test)).item()
        if val_loss < best_val_loss - 1e-5:
            best_val_loss  = val_loss
            best_enc_state = copy.deepcopy(encoder.state_dict())
            best_cls_state = copy.deepcopy(classifier.state_dict())
            patience_cnt   = 0
        else:
            patience_cnt += 1
            if patience_cnt >= hp.early_patience:
                break

    encoder.load_state_dict(best_enc_state)
    classifier.load_state_dict(best_cls_state)

    proba_s1_te = _get_proba(encoder, classifier, X_te)
    acc_s1  = accuracy_score(y_test, (proba_s1_te >= 0.5).astype(int))
    eo_s1   = max((compute_eo(y_test, (proba_s1_te >= 0.5).astype(int), s) for s in s_test), default=0.0)
    dp_s1   = max((compute_dp((proba_s1_te >= 0.5).astype(int), s) for s in s_test), default=0.0)
    if verbose:
        print(f"  Stage1 acc={acc_s1:.4f} EO={eo_s1:.4f} DP={dp_s1:.4f}")

    # ── Diagnostic checkpoint: after Stage 1 ─────────────────
    diag_checkpoints = [
        {'stage': 'Stage1 (no fairness)', 'acc': acc_s1, 'eo': eo_s1, 'dp': dp_s1,
         'tau': 0.5, 'note': 'base model quality'},
    ]

    enc_lr = hp.lr * hp.encoder_lr_factor if hp.encoder_lr_factor > 0 else hp.lr * 0.1
    adv_lr = hp.lr * hp.adversary_lr_factor

    opt_enc = optim.Adam(list(encoder.parameters()) + list(classifier.parameters()),
                         lr=enc_lr, weight_decay=1e-5)
    opt_adv = optim.Adam(adversary.parameters(), lr=adv_lr, weight_decay=1e-5)

    primary_metric = eo_s1 if mode in ('eo_only', 'both') else dp_s1
    lambda_sched = AdaptiveLambdaScheduler(
        start=hp.lambda_adv_start,
        max_val=hp.lambda_adv_max,
        warmup=hp.lambda_warmup_epochs,
        adaptive=hp.adaptive_lambda,
        clip=hp.adaptive_lambda_clip,
    )
    lambda_sched.update_metric(primary_metric)

    best_s2_score  = np.inf
    best_s2_pred   = (proba_s1_te >= 0.5).astype(int)
    acc_budget     = original_baseline_acc - hp.stage2_max_acc_drop

    # Track best encoder for diagnostics
    best_s2_enc_state = copy.deepcopy(encoder.state_dict())
    best_s2_cls_state = copy.deepcopy(classifier.state_dict())

    for epoch in range(hp.fairness_epochs):
        lam = lambda_sched.get(epoch)

        encoder.train(); classifier.train(); adversary.train()
        for batch in dl_tr:
            xb      = batch[0]; yb = batch[1]
            sb_list = list(batch[2:])
            batch_idx = torch.randint(0, len(bbn_prior_t), (len(xb),))

            # Adversary update
            z_det   = encoder(xb).detach()
            adv_out = adversary(z_det)
            loss_adv = sum(bce_unw(ao, sb.to(DEVICE))
                           for ao, sb in zip(adv_out, sb_list)) / max(1, n_sens)
            opt_adv.zero_grad(); loss_adv.backward(); opt_adv.step()

            # Encoder + classifier update
            z       = encoder(xb)
            logit   = classifier(z)
            prob    = torch.sigmoid(logit)
            adv_out = adversary(z)

            loss_cls  = bce(logit, yb) * hp.cls_loss_weight_s2

            # Adversarial confusion (always included; direction depends on mode)
            loss_conf = sum(bce_unw(ao, sb.to(DEVICE))
                            for ao, sb in zip(adv_out, sb_list)) / max(1, n_sens)

            # ── Mode-aware direct fairness loss ───────────────
            loss_fair_direct = torch.tensor(0.0, device=DEVICE)

            if mode in ('eo_only', 'both'):
                # Direct EO loss (primary in eo_only mode)
                eo_weight = hp.eo_loss_weight * (2.0 if mode == 'eo_only' else 1.0)
                for sb in sb_list:
                    loss_fair_direct = loss_fair_direct + differentiable_eo_loss(
                        prob, yb, sb.to(DEVICE), temp=hp.eo_smooth_temp)
                loss_fair_direct = loss_fair_direct * eo_weight / max(1, n_sens)

            if mode in ('dp_only', 'both'):
                # Direct DP loss (primary in dp_only mode)
                dp_weight = hp.eo_loss_weight * (2.0 if mode == 'dp_only' else 0.5)
                loss_dp = torch.tensor(0.0, device=DEVICE)
                for sb in sb_list:
                    loss_dp = loss_dp + differentiable_dp_loss(
                        prob, sb.to(DEVICE), temp=hp.eo_smooth_temp)
                loss_fair_direct = loss_fair_direct + loss_dp * dp_weight / max(1, n_sens)

            loss_bbn_s2 = torch.tensor(0.0, device=DEVICE)
            if hp.use_bbn and bbn_model is not None and bbn_model.fitted:
                loss_bbn_s2 = F.binary_cross_entropy(
                    prob, bbn_fair_target_t[batch_idx].clamp(0.05, 0.95))

            loss_enc = (loss_cls
                        - lam * loss_conf
                        + loss_fair_direct
                        + hp.bbn_adv_weight * loss_bbn_s2)
            opt_enc.zero_grad(); loss_enc.backward()
            nn.utils.clip_grad_norm_(
                list(encoder.parameters()) + list(classifier.parameters()), 1.0)
            opt_enc.step()

        if (epoch + 1) % 10 == 0 or epoch == hp.fairness_epochs - 1:
            proba_te = _get_proba(encoder, classifier, X_te)
            acc_now  = accuracy_score(y_test, (proba_te >= 0.5).astype(int))
            eo_now   = max((compute_eo(y_test, (proba_te >= 0.5).astype(int), s)
                            for s in s_test), default=0.0)
            dp_now   = max((compute_dp((proba_te >= 0.5).astype(int), s)
                            for s in s_test), default=0.0)

            primary_now = eo_now if mode in ('eo_only', 'both') else dp_now
            lambda_sched.update_metric(primary_now)

            fl_eo = floor2(eo_now); fl_dp = floor2(dp_now)
            if mode == 'eo_only':
                score = (max(0.0, original_baseline_acc - acc_now) if fl_eo == 0.0
                         else fl_eo * 100 + eo_now * 0.3)
            elif mode == 'dp_only':
                score = (max(0.0, original_baseline_acc - acc_now) if fl_dp == 0.0
                         else fl_dp * 100 + dp_now * 0.3)
            else:
                score = (max(0.0, original_baseline_acc - acc_now) + fl_dp * 0.5
                         if fl_eo == 0.0 else fl_eo * 100 + fl_dp * 5 + (eo_now + dp_now) * 0.3)

            if score < best_s2_score and acc_now >= acc_budget:
                best_s2_score  = score
                best_s2_pred   = (proba_te >= 0.5).astype(int)
                best_s2_enc_state = copy.deepcopy(encoder.state_dict())
                best_s2_cls_state = copy.deepcopy(classifier.state_dict())
            if verbose and (epoch + 1) % 50 == 0:
                print(f"  S2 ep{epoch+1}: acc={acc_now:.4f} EO={eo_now:.4f} "
                      f"DP={dp_now:.4f} lam={lam:.2f}")

    encoder.load_state_dict(best_s2_enc_state)
    classifier.load_state_dict(best_s2_cls_state)

    proba_tr_final = _get_proba(encoder, classifier, X_tr)
    proba_te_final = _get_proba(encoder, classifier, X_te)

    # Diagnostic: after Stage 2 training
    acc_s2 = accuracy_score(y_test, (proba_te_final >= 0.5).astype(int))
    eo_s2  = max((compute_eo(y_test, (proba_te_final >= 0.5).astype(int), s) for s in s_test), default=0.0)
    dp_s2  = max((compute_dp((proba_te_final >= 0.5).astype(int), s) for s in s_test), default=0.0)
    diag_checkpoints.append({'stage': 'Stage2 (fairness training)',
                              'acc': acc_s2, 'eo': eo_s2, 'dp': dp_s2, 'tau': 0.5,
                              'note': f'best checkpoint over {hp.fairness_epochs} epochs'})

    if hp.use_isotonic:
        proba_te_cal, proba_tr_cal = calibrate_proba(proba_tr_final, y_train, proba_te_final)
    else:
        proba_te_cal = proba_te_final
        proba_tr_cal = proba_tr_final

    # ── NEW: Group calibration ─────────────────────────────────
    # Only active in eo_only mode; uses primary sensitive attribute
    if mode == 'eo_only' and hp.group_calib_alpha > 0 and len(s_train) > 0:
        gc_model = GroupCalibrator(alpha=hp.group_calib_alpha)
        gc_model.fit(proba_tr_cal, y_train, s_train[0])
        proba_te_cal_gc = gc_model.transform_test(proba_te_cal, s_test[0])
    else:
        proba_te_cal_gc = proba_te_cal

    # Diagnostic: after calibration
    acc_cal = accuracy_score(y_test, (proba_te_cal_gc >= 0.5).astype(int))
    eo_cal  = max((compute_eo(y_test, (proba_te_cal_gc >= 0.5).astype(int), s) for s in s_test), default=0.0)
    dp_cal  = max((compute_dp((proba_te_cal_gc >= 0.5).astype(int), s) for s in s_test), default=0.0)
    diag_checkpoints.append({'stage': 'Post-calibration + GroupCalib',
                              'acc': acc_cal, 'eo': eo_cal, 'dp': dp_cal, 'tau': 0.5,
                              'note': f'group_calib_alpha={hp.group_calib_alpha:.2f}'})

    if verbose:
        print(f"  Pre-sweep: acc={acc_cal:.4f} EO={eo_cal:.4f} DP={dp_cal:.4f}")

    y_pred_final, tau_used = joint_threshold_sweep(
        y_test, proba_te_cal_gc, s_test, hp, original_baseline_acc, mode=mode)

    acc_final  = accuracy_score(y_test, y_pred_final)
    fair_final = compute_fairness_metrics(y_test, y_pred_final, s_dict)

    # Diagnostic: after sweep
    eo_sw = max(v for k, v in fair_final.items() if k.endswith('_eo'))
    dp_sw = max(v for k, v in fair_final.items() if k.endswith('_dp'))
    diag_checkpoints.append({'stage': f'Post-sweep (tau={tau_used:.3f})',
                              'acc': acc_final, 'eo': eo_sw, 'dp': dp_sw, 'tau': tau_used,
                              'note': (f'constrained to [{hp.sweep_tau_min:.2f},{hp.sweep_tau_max:.2f}]'
                                       if hp.sweep_tau_min > 0.05 else 'full range used')})

    result = {"acc": acc_final, **fair_final}

    if verbose:
        drop = original_baseline_acc - acc_final
        print(f"  Final: acc={acc_final:.4f} drop={drop:+.4f} "
              f"EO_fl={floor2(eo_sw):.2f} DP_fl={floor2(dp_sw):.2f} tau={tau_used:.3f}")

    # Print diagnostics every run
    run_diagnostics(dataset_key, original_baseline_acc, diag_checkpoints, result)

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return result


print("train_model defined.")

_PREPROCESS = {
    "adult":     preprocess_adult_for_fair_bbn,
    "compas":    preprocess_compas_for_fair_bbn,
    "german":    preprocess_german_for_fair_bbn,
    "bank":      preprocess_bank_for_fair_bbn,
    "lawschool": preprocess_lawschool_for_fair_bbn,
    "hospital":  preprocess_diabetes_hospital_for_fair_bbn,
}


def _get_baseline_acc(data: dict) -> float:
    X_tr = to_dense(data['X_train_nn'])
    X_te = to_dense(data['X_test_nn'])
    mlp  = MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=300,
                         random_state=SEED, early_stopping=True, verbose=False)
    mlp.fit(X_tr, data['y_train'])
    return float(accuracy_score(data['y_test'], mlp.predict(X_te)))


def run_dataset(dataset_key: str, n_runs: int = 3, verbose: bool = True,
                mode: str = PIPELINE_MODE):
    assert dataset_key in _PREPROCESS, f"Unknown dataset '{dataset_key}'"
    data = _PREPROCESS[dataset_key]()
    hp   = DATASET_HPARAMS[dataset_key]
    bacc = _get_baseline_acc(data)
    print(f"\n  {dataset_key.upper()}  baseline_acc={bacc:.4f}  mode={mode}")

    results = []
    for run in range(n_runs):
        try:
            r = train_model(data, dataset_key, hp,
                            original_baseline_acc=bacc, verbose=verbose,
                            mode=mode)
            results.append(r)
            eo_v = max((abs(v) for k, v in r.items() if k.endswith("_eo")), default=1.0)
            dp_v = max((abs(v) for k, v in r.items() if k.endswith("_dp")), default=1.0)
            tag  = ("ZERO" if (floor2(eo_v)==0.0 and floor2(dp_v)==0.0) else
                    "EO=0" if floor2(eo_v)==0.0 else
                    "DP=0" if floor2(dp_v)==0.0 else "    ")
            print(f"  [{tag}] run {run}: acc={r.get('acc',0):.4f} "
                  f"EO_fl={floor2(eo_v):.2f} DP_fl={floor2(dp_v):.2f}")
        except Exception as e:
            print(f"  run {run} error: {e}")
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    return results, bacc


def run_all_datasets(
    datasets=None,
    n_runs:              int  = 3,
    verbose:             bool = True,
    continue_on_error:   bool = True,
    mode:                str  = PIPELINE_MODE,
):
    all_valid = list(_PREPROCESS.keys())
    if datasets is None:
        datasets = all_valid

    print("="*65)
    print(f"  FAIRNESS PIPELINE v3  —  mode: {mode.upper()}")
    if mode == 'eo_only':
        print(f"  Primary target: floor(EO)=0.00 with minimal accuracy drop")
        print(f"  DP is tracked/reported but NOT optimised")
    elif mode == 'dp_only':
        print(f"  Primary target: floor(DP)=0.00 with minimal accuracy drop")
        print(f"  EO is tracked/reported but NOT optimised")
    else:
        print(f"  Primary target: floor(EO)=0.00, secondary: floor(DP)=0.00")
    print(f"  Device: {DEVICE}  |  Datasets: {datasets}")
    print("="*65)

    results_cache = {}
    failed        = []

    for i, ds in enumerate(datasets, 1):
        print(f"\n{'─'*65}")
        print(f"  [{i}/{len(datasets)}] {ds.upper()}")
        print(f"{'─'*65}")
        try:
            res, bacc = run_dataset(ds, n_runs=n_runs, verbose=verbose, mode=mode)
            results_cache[ds] = (res, bacc)
        except Exception as e:
            print(f"  FAILED {ds}: {e}")
            failed.append(ds)
            if not continue_on_error:
                raise

    print(f"\n  Done: {[d for d in datasets if d not in failed]}")
    print(f"  Failed: {failed}")
    _print_final_table(results_cache, mode=mode)
    return results_cache


def _print_final_table(results_cache: dict, mode: str = PIPELINE_MODE):
    import json
    print(f"\n{'='*70}")
    print(f"  FINAL RESULTS  (mode={mode})")
    print(f"  {'Dataset':12} {'Base':>7} {'Ours':>7} {'Drop':>7} "
          f"{'EO_fl':>6} {'DP_fl':>6}  Status  Note")
    print("  " + "─"*60)

    for ds, (rr, bacc) in results_cache.items():
        if not rr:
            print(f"  {ds.upper():12} no results")
            continue
        keys = rr[0].keys()
        avg  = {k: float(np.mean([r[k] for r in rr if k in r]))
                for k in keys if isinstance(rr[0].get(k), (int, float, np.floating))}
        eo_vals = [abs(avg[k]) for k in avg if k.endswith("_eo")]
        dp_vals = [abs(avg[k]) for k in avg if k.endswith("_dp")]
        max_eo  = max(eo_vals, default=1.0)
        max_dp  = max(dp_vals, default=1.0)
        drop    = bacc - avg.get("acc", 0.0)
        ef, df  = floor2(max_eo), floor2(max_dp)
        full_tag = "ZERO " if (ef==0.0 and df==0.0) else \
                   "EO=0 " if ef==0.0 else \
                   "DP=0 " if df==0.0 else "FAIL "
        # Note if drop is concerning
        note = "HIGH DROP" if drop > 0.07 else ""
        print(f"  {ds.upper():12} {bacc:7.4f} {avg.get('acc',0):7.4f} "
              f"{drop:+7.4f} {ef:6.2f} {df:6.2f}  {full_tag}  {note}")

    out_name = f"final_results_{mode}.json"
    out = os.path.join(RESULTS_DIR, out_name)
    with open(out, "w") as f:
        json.dump({
            "mode": mode,
            "results": {ds: {
                "baseline_acc": bacc,
                "our_acc": float(np.mean([r.get("acc",0) for r in rr])) if rr else 0,
                "eo": float(np.mean([max(abs(v) for k,v in r.items() if k.endswith('_eo'))
                                     for r in rr])) if rr else 1.0,
                "dp": float(np.mean([max(abs(v) for k,v in r.items() if k.endswith('_dp'))
                                     for r in rr])) if rr else 1.0,
                "n_runs": len(rr),
            } for ds, (rr, bacc) in results_cache.items()}
        }, f, indent=2)
    print(f"\n  Saved → {out}")


print("Pipeline functions defined.")
print(f"\nTo run EO-only:  set PIPELINE_MODE = 'eo_only' at the top")
print(f"To run DP-only:  set PIPELINE_MODE = 'dp_only' at the top")
print(f"To run both:     set PIPELINE_MODE = 'both' at the top")
print(f"Current mode:    {PIPELINE_MODE}")


results_cache = run_all_datasets(
    datasets=['adult', 'compas', 'german', 'bank', 'lawschool', 'hospital'],
    n_runs=3,
    verbose=True,
    continue_on_error=True,
    mode=PIPELINE_MODE,
)

print("\nDone.")

/usr/local/lib/python3.12/dist-packages/pgmpy/estimators/__init__.py:4: FutureWarning: `pgmpy.estimators.StructureScore` is deprecated and will be removed in v1.3.0. Use `pgmpy.structure_score` instead.
  from .StructureScore import (


Device: cuda  |  CUDA: True
Pipeline mode: eo_only
Hyperparams defined. Mode: eo_only
Utility helpers defined.
All 6 preprocessing functions defined.
Neural network modules defined.
Threshold sweep defined.
train_model defined.
Pipeline functions defined.

To run EO-only:  set PIPELINE_MODE = 'eo_only' at the top
To run DP-only:  set PIPELINE_MODE = 'dp_only' at the top
To run both:     set PIPELINE_MODE = 'both' at the top
Current mode:    eo_only
  FAIRNESS PIPELINE v3  —  mode: EO_ONLY
  Primary target: floor(EO)=0.00 with minimal accuracy drop
  DP is tracked/reported but NOT optimised
  Device: cuda  |  Datasets: ['adult', 'compas', 'german', 'bank', 'lawschool', 'hospital']

─────────────────────────────────────────────────────────────────
  [1/6] ADULT
─────────────────────────────────────────────────────────────────

  ADULT  baseline_acc=0.8618  mode=eo_only
  Stage1 acc=0.8203 EO=0.1817 DP=0.2896
  S2 ep50: acc=0.7351 EO=0.0395 DP=0.1013 lam=4.50
  S2 ep100: acc=0.7618 EO=0.0

can the above version be improved for eo. does that need any hyperparam tuning? maybe remove post calibration and sweeping thing. better focus on reducing the accuracy loss. eo only, dp only. should it be combined?

The key thing is to keep the architecture identical and only change the loss/scoring functions, so the comparison is controlled. (while running dp script)